<!--nav--> [🗺 Learning path](README.md) · **36/39** · ◀ [Serving Mixture-of-Experts](./MoE_Serving_Expert_Parallelism.ipynb) · [Production Hardening](./Production_Hardening_Reliability.ipynb) ▶

# RAG & Agent Serving Patterns: Caches, Cascades, and Prompt Layout

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/RAG_Agent_Serving_Patterns.ipynb)

Every notebook so far treated requests as independent. Real applications aren't: a RAG system sends
the same documents over and over, and an agent re-sends its **entire growing history** on every
single step. That changes the serving problem from "make one request fast" to "**stop doing the same
work repeatedly**".

The wins here are the largest in the whole track — and they're almost all free.

| Part | What you'll learn |
|---|---|
| **1** | The shape of agent traffic: quadratic prefill, and why it sneaks up on you |
| **2** | **Prompt layout** — the ordering rule that decides your cache hit rate |
| **3** | The cache hierarchy: GPU blocks → CPU → distributed KV store |
| **4** | **Model cascades**: small-model-first routing, simulated with cost and quality |
| **5** | **Semantic caching** — the one with a sharp edge |
| **6** | Putting it together: an agent-workload simulator comparing five strategies |
| **7** | The checklist |

**Runs on:** any CPU.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · Agent traffic is quadratic

An agent loop looks innocent:

```
turn 1:  [system][tools][history₀] + question  →  answer₁
turn 2:  [system][tools][history₀][q₁][a₁] + question  →  answer₂
turn 3:  [system][tools][history₀][q₁][a₁][q₂][a₂] + question  →  answer₃
          └──────────── grows every turn ────────────┘
```

Each turn's prompt contains every previous turn. Prefill work over an N-turn session is therefore
**O(N²)** while the useful output is O(N). This is why "our agent got slow as conversations got
longer" is the single most common LLM-application performance complaint.

In [ ]:
def session_prefill(n_turns, base_tokens, per_turn_tokens, cached=False):
    if cached:
        # only the genuinely-new tokens are prefilled each turn
        return base_tokens + per_turn_tokens * n_turns
    return sum(base_tokens + per_turn_tokens * t for t in range(n_turns))

BASE, PER_TURN = 2000, 400        # system+tools prompt, and tokens added per turn
print(f"System+tools prompt: {BASE} tokens · each turn adds {PER_TURN} tokens\n")
print(f"{'turns':>7}{'prefill (no cache)':>20}{'prefill (cached)':>18}{'wasted':>10}{'ratio':>8}")
print("-" * 66)
for n in (1, 5, 10, 20, 50, 100):
    a = session_prefill(n, BASE, PER_TURN)
    b = session_prefill(n, BASE, PER_TURN, cached=True)
    print(f"{n:>7}{a:>20,}{b:>18,}{a-b:>10,}{a/b:>7.1f}x")

print("\nAt 50 turns you are doing ~11x more prefill than necessary.")
print("The output tokens - the thing the user actually wanted - grew only linearly.")
print("\nAnd note WHERE the cost lands: prefill is compute-bound (nb 21), so this shows up")
print("as TTFT growing every turn, plus prefill stealing decode slots from other users (nb 26).")

## Part 2 · Prompt layout: the ordering rule

Prefix caching (nb 22) matches **from the start of the prompt, block by block**. One differing token
early invalidates everything after it. So the rule is simple and absolute:

```
   ✅ GOOD                                  ❌ BAD
   [ system prompt      ] static            [ timestamp: 14:32:07 ]  ← changes every request!
   [ tool definitions   ] static            [ system prompt       ]
   [ retrieved docs     ] semi-static       [ user question       ]
   [ conversation       ] append-only       [ retrieved docs      ]
   [ user question      ] volatile          [ conversation        ]
   → cache hit on everything above          → cache hit on NOTHING
     the question                             (0% reuse, every single time)
```

**Sort your prompt by volatility: most static first, most volatile last.** Let's measure what
different layouts actually yield:

In [ ]:
SEGMENTS = [                  # (name, tokens, how often it changes)
    ("system prompt",     800,  "never"),
    ("tool definitions", 1200,  "never"),
    ("retrieved docs",   3000,  "per-session"),
    ("conversation",     2000,  "per-turn (append-only)"),
    ("user question",     150,  "per-turn"),
    ("timestamp/ids",      20,  "per-request"),
]

def cache_hit_tokens(order):
    # Cached prefix = the leading run of segments that haven't changed since last request.
    STABLE = {"never", "per-session", "per-turn (append-only)"}
    hit = 0
    for name, toks, churn in order:
        if churn in STABLE:
            hit += toks
        else:
            break                      # first volatile segment ends the reusable prefix
    return hit

LAYOUTS = {
    "volatility-sorted (best)": [SEGMENTS[0], SEGMENTS[1], SEGMENTS[2], SEGMENTS[3],
                                 SEGMENTS[4], SEGMENTS[5]],
    "timestamp first (common bug)": [SEGMENTS[5], SEGMENTS[0], SEGMENTS[1], SEGMENTS[2],
                                     SEGMENTS[3], SEGMENTS[4]],
    "question before docs":     [SEGMENTS[0], SEGMENTS[1], SEGMENTS[4], SEGMENTS[2],
                                 SEGMENTS[3], SEGMENTS[5]],
    "docs last (RAG default)":  [SEGMENTS[0], SEGMENTS[1], SEGMENTS[3], SEGMENTS[4],
                                 SEGMENTS[2], SEGMENTS[5]],
}
total = sum(s[1] for s in SEGMENTS)
print(f"Total prompt: {total:,} tokens\n")
print(f"{'layout':<32}{'cacheable prefix':>18}{'hit rate':>10}{'prefill/req':>13}")
print("-" * 74)
for name, order in LAYOUTS.items():
    hit = cache_hit_tokens(order)
    print(f"{name:<32}{hit:>18,}{hit/total:>9.0%}{total-hit:>13,}")

print("\nThe 'timestamp first' row is not a strawman - injecting the current time or a")
print("request ID at the top of a system prompt is one of the most common ways teams")
print("accidentally disable prefix caching entirely. It costs 0% -> 96% of your reuse.")
print("\nDiagnosis (nb 26): if `Prefix cache hit rate` is ~0% on repetitive traffic,")
print("look at the FIRST 50 tokens of your prompt before you look at anything else.")

## Part 3 · The cache hierarchy

Prefix caching in GPU memory is level one. Production systems build a hierarchy, exactly like CPU
caches:

| Level | Where | Capacity | Latency to reuse | Notes |
|---|---|---|---|---|
| **L1** | GPU KV blocks | GB | free (already there) | vLLM's automatic prefix cache (nb 22) |
| **L2** | CPU RAM | 100s of GB | PCIe transfer | offload; still far cheaper than re-prefill |
| **L3** | Local NVMe | TB | disk read | for very large document sets |
| **L4** | Distributed KV store | ∞ | network | LMCache, Mooncake — share across replicas (nb 29) |

The economics of the decision are simple: **re-transferring KV is cheaper than re-computing it**
whenever the transfer is faster than prefill. Let's find the crossover:

In [ ]:
def reuse_vs_recompute(prompt_tokens, kv_kb_per_token=128, prefill_tps=9000,
                       links=(("PCIe 4.0 (CPU offload)", 25e9),
                              ("NVMe read", 5e9),
                              ("100GbE (distributed)", 12.5e9),
                              ("InfiniBand 400G", 50e9))):
    recompute_s = prompt_tokens / prefill_tps
    kv_bytes = prompt_tokens * kv_kb_per_token * 1024
    out = {"recompute_s": recompute_s, "kv_gb": kv_bytes / 1e9}
    for name, bps in links:
        out[name] = kv_bytes / bps
    return out

print("Reuse a cached prefix vs re-prefilling it from scratch:\n")
print(f"{'prompt tokens':>14}{'recompute':>12}{'PCIe':>10}{'NVMe':>10}{'100GbE':>10}{'IB 400G':>10}")
print("-" * 68)
for n in (1024, 4096, 16384, 65536, 262144):
    r = reuse_vs_recompute(n)
    print(f"{n:>14,}{r['recompute_s']*1000:>10.0f}ms"
          f"{r['PCIe 4.0 (CPU offload)']*1000:>9.0f}ms{r['NVMe read']*1000:>9.0f}ms"
          f"{r['100GbE (distributed)']*1000:>9.0f}ms{r['InfiniBand 400G']*1000:>9.0f}ms")

print("\nRead across a row: any transfer number BELOW the recompute number is a win.")
print("PCIe offload beats recomputation at every size here - CPU offload is nearly always")
print("worth enabling. Slow links (NVMe, plain Ethernet) only pay off for large prefixes,")
print("which is exactly the regime where distributed KV stores are deployed.")
print("\nCaveat: this ignores the cost of MISSING. A cache lookup that misses adds latency")
print("to the recompute path - so hit rate has to be decent for the hierarchy to pay.")

## Part 4 · Model cascades

The other big idea: **not every request needs your biggest model**. A cascade tries a small model
first and escalates only when it isn't confident (or when a verifier rejects the answer).

```
request ──► small model ──► confident?  ──yes──► return (cheap, fast)
                                │
                                no
                                ▼
                          large model ──► return (expensive, better)
```

The economics hinge on two numbers: the **escalation rate** and the **cost ratio**. The trap is that
escalated requests pay *both* models, so a cascade with a high escalation rate is **worse than just
using the big model**.

In [ ]:
def cascade(escalation_rate, small_cost=1.0, large_cost=20.0,
            small_quality=0.78, large_quality=0.93, small_latency=0.4, large_latency=2.0):
    # Escalated requests pay for BOTH models (the small attempt is sunk cost).
    cost = small_cost + escalation_rate * large_cost
    latency = small_latency + escalation_rate * large_latency
    # Assume escalation preferentially catches the cases the small model would have failed.
    quality = small_quality + escalation_rate * (large_quality - small_quality) / max(escalation_rate, 1e-9) \
              if False else (1 - escalation_rate) * small_quality + escalation_rate * large_quality
    return {"escalation": escalation_rate, "cost": cost, "latency": latency, "quality": quality}

large_only = {"cost": 20.0, "latency": 2.0, "quality": 0.93}
print(f"small model = 1 unit, large = 20 units. Large-model-only baseline: "
      f"cost {large_only['cost']}, quality {large_only['quality']:.0%}\n")
print(f"{'escalation rate':>16}{'cost':>9}{'vs large-only':>15}{'avg latency':>13}{'quality':>10}")
print("-" * 66)
for rate in (0.0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0):
    c = cascade(rate)
    print(f"{rate:>15.0%}{c['cost']:>9.1f}{c['cost']/large_only['cost']:>14.0%}"
          f"{c['latency']:>12.2f}s{c['quality']:>10.0%}")

breakeven = (large_only["cost"] - 1.0) / 20.0
print(f"\nBreak-even escalation rate: {breakeven:.0%}. Above that, the cascade costs MORE")
print("than just calling the large model every time - because you paid the small model for nothing.")
print("\nSo a cascade is only worth building if you can genuinely route most traffic to the")
print("small model. Measure your escalation rate BEFORE building the routing logic.")
print("\nWhat makes escalation decisions work in practice:")
print("  - logprob/entropy thresholds on the small model's answer")
print("  - a cheap verifier (rules, a classifier, or structured-output validation - nb 28)")
print("  - task-based routing: classify the request type up front, skip the small attempt entirely")

## Part 5 · Semantic caching (handle with care)

The most aggressive cache: embed the incoming request, and if it's *similar enough* to a previous
one, return the stored **response** without running the model at all. Hit rates on FAQ-shaped
traffic can be very high, and the saving is 100% of the request.

The sharp edge: **"similar" is not "the same"**. A threshold loose enough to be useful will
eventually serve the answer to a *different* question. Consider:

> *"Can I cancel my order?"* vs *"Can I cancel my order after it ships?"*

These embed very closely and have different answers. Let's model the trade explicitly, because the
failure is silent:

In [ ]:
def semantic_cache(threshold, n=10000, seed=1):
    # Model: query pairs have a similarity score; "same intent" pairs score higher,
    # but the distributions OVERLAP - which is the entire problem.
    rng = random.Random(seed)
    hits = wrong = miss = 0
    for _ in range(n):
        same_intent = rng.random() < 0.35                    # 35% are genuine repeats
        sim = rng.gauss(0.93, 0.04) if same_intent else rng.gauss(0.80, 0.07)
        if sim >= threshold:
            if same_intent: hits += 1
            else: wrong += 1                                  # served the WRONG cached answer
        else:
            miss += 1
    return {"threshold": threshold, "hit_rate": hits / n,
            "wrong_rate": wrong / n, "miss_rate": miss / n,
            "precision": hits / max(hits + wrong, 1)}

print(f"{'threshold':>10}{'cache hits':>12}{'WRONG answers':>15}{'misses':>9}{'precision':>11}")
print("-" * 60)
for t in (0.80, 0.85, 0.88, 0.90, 0.92, 0.95, 0.98):
    r = semantic_cache(t)
    flag = "  ⚠" if r["wrong_rate"] > 0.01 else ""
    print(f"{t:>10.2f}{r['hit_rate']:>11.1%}{r['wrong_rate']:>14.1%}{r['miss_rate']:>9.1%}"
          f"{r['precision']:>10.1%}{flag}")

print("\nThere is no threshold that gives you high hit rate AND zero wrong answers,")
print("because the similarity distributions overlap. You are choosing an error budget.")
print("\nUse semantic caching when: answers are stable, the domain is narrow (FAQ/support),")
print("and a wrong-but-plausible answer is annoying rather than harmful.")
print("Do NOT use it for: anything personalized, transactional, time-sensitive, or safety-relevant.")
print("\nSafer alternative that captures much of the win with none of the risk:")
print("  exact-match caching on normalized text + PREFIX caching (nb 22) for the rest.")
print("  Prefix caching is semantically lossless - it reuses computation, never conclusions.")

## Part 6 · Putting it together

Five strategies, one realistic agent workload. Each strategy is cumulative with the previous one, so
you can see what each layer actually adds.

In [ ]:
random.seed(11)

def agent_workload(n_sessions=200, turns_mean=8, base=2000, per_turn=400,
                   n_doc_sets=20, doc_tokens=3000):
    sessions = []
    for s in range(n_sessions):
        turns = max(1, int(random.gauss(turns_mean, 3)))
        sessions.append({"id": s, "turns": turns, "docs": random.randrange(n_doc_sets)})
    return sessions

SESSIONS = agent_workload()
BASE, PER_TURN, DOC = 2000, 400, 3000
PREFILL_TPS, DECODE_TPS, OUT_TOKENS = 9000, 25, 180

def evaluate(strategy):
    prefill = decode = 0
    doc_cache_seen = set()
    for s in SESSIONS:
        for t in range(s["turns"]):
            history = PER_TURN * t
            full_prompt = BASE + DOC + history + 150

            if strategy == "no caching":
                prefill += full_prompt
            elif strategy == "+ prefix cache (bad layout)":
                prefill += full_prompt                      # volatile token at the front kills it
            elif strategy == "+ prefix cache (good layout)":
                prefill += (full_prompt if t == 0 else PER_TURN + 150)
            elif strategy == "+ cross-session doc cache":
                if t == 0:
                    prefill += (BASE + DOC + 150) if s["docs"] not in doc_cache_seen else (BASE + 150)
                    doc_cache_seen.add(s["docs"])
                else:
                    prefill += PER_TURN + 150
            elif strategy == "+ cascade (30% escalate)":
                if t == 0:
                    prefill += (BASE + DOC + 150) if s["docs"] not in doc_cache_seen else (BASE + 150)
                    doc_cache_seen.add(s["docs"])
                else:
                    prefill += PER_TURN + 150
            decode += OUT_TOKENS

    gpu_seconds = prefill / PREFILL_TPS + decode / DECODE_TPS
    if strategy == "+ cascade (30% escalate)":
        # 70% of turns answered by a model ~8x cheaper; 30% pay both
        gpu_seconds = gpu_seconds * (0.70 / 8 + 0.30 * (1 + 1 / 8))
    return {"strategy": strategy, "prefill": prefill, "decode": decode, "gpu_s": gpu_seconds}

STRATEGIES = ["no caching", "+ prefix cache (bad layout)", "+ prefix cache (good layout)",
              "+ cross-session doc cache", "+ cascade (30% escalate)"]
rows = [evaluate(s) for s in STRATEGIES]
base_row = rows[0]

turns_total = sum(s["turns"] for s in SESSIONS)
print(f"{len(SESSIONS)} sessions, {turns_total:,} turns total, {DOC}-token doc sets\n")
print(f"{'strategy':<32}{'prefill tokens':>16}{'GPU-seconds':>13}{'vs baseline':>13}")
print("-" * 76)
for r in rows:
    print(f"{r['strategy']:<32}{r['prefill']:>16,}{r['gpu_s']:>13,.0f}"
          f"{r['gpu_s']/base_row['gpu_s']:>12.0%}")

print(f"\nTotal saving from baseline to full stack: "
      f"{1 - rows[-1]['gpu_s']/base_row['gpu_s']:.0%} of GPU time.")
print("Note that the SECOND row saves nothing: enabling prefix caching with a bad prompt")
print("layout is a no-op, which is why teams report 'we turned it on and nothing happened'.")

In [ ]:
JS = r'''
const M = {top: 18, right: 150, bottom: 44, left: 250};
const iw = W - M.left - M.right, ih = data.length * 40;
const svg = root.append("svg").attr("width",W).attr("height",ih+M.top+M.bottom)
    .append("g").attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleLinear().domain([0, d3.max(data,d=>d.gpu_s)*1.12]).range([0,iw]);
const y = d3.scaleBand().domain(data.map(d=>d.strategy)).range([0,ih]).padding(0.25);
svg.append("g").attr("transform",`translate(0,${ih})`).call(d3.axisBottom(x).ticks(6,"~s"));
svg.append("g").call(d3.axisLeft(y).tickSize(0)).select(".domain").remove();
svg.append("text").attr("x",iw/2).attr("y",ih+36).attr("text-anchor","middle")
   .style("font-size","12px").text("GPU-seconds for the whole workload (lower is better)");
const base = data[0].gpu_s;
svg.selectAll("b").data(data).join("rect")
   .attr("x",0).attr("y",d=>y(d.strategy)).attr("height",y.bandwidth()).attr("rx",3)
   .attr("width",d=>x(d.gpu_s))
   .attr("fill",(d,i)=> i===0 ? "#90a4ae" : (d.gpu_s/base > 0.98 ? "#ef9a9a" : "#43a047"))
   .append("title").text(d=>`${d.strategy}: ${d3.format(",.0f")(d.gpu_s)} GPU-s`);
svg.selectAll("t").data(data).join("text")
   .attr("x",d=>x(d.gpu_s)+8).attr("y",d=>y(d.strategy)+y.bandwidth()/2+4)
   .style("font-size","11px").style("fill","#455a64")
   .text(d=>`${d3.format(",.0f")(d.gpu_s)} GPU-s  (${d3.format(".0%")(d.gpu_s/base)})`);
'''
show_d3(JS, rows, height=len(rows)*40 + 70)

## Part 7 · The checklist

**Prompt construction**
- [ ] Segments ordered **static → volatile**; nothing dynamic in the first blocks
- [ ] No timestamps, request IDs, or random nonces near the top
- [ ] Retrieved documents placed **before** the question, and kept byte-identical between turns
- [ ] Tool definitions stable (don't re-serialize dicts in nondeterministic order!)

**Caching**
- [ ] Prefix caching on (default in vLLM V1) and **hit rate verified in metrics** (nb 26)
- [ ] Session affinity / prefix-aware routing so hits land on the right replica (nb 29)
- [ ] CPU offload enabled if your prefixes are large (Part 3)
- [ ] Exact-match response cache before considering semantic caching
- [ ] If semantic caching: a measured error budget, a kill switch, and no personalized content

**Model strategy**
- [ ] Escalation rate **measured** before building a cascade (Part 4)
- [ ] Structured output for tool calls so failures are parse-free (nb 28)
- [ ] Speculative decoding considered — agent output is often highly predictable (nb 24)

**Watch**
- [ ] Prefix hit rate, token amplification, TTFT vs turn index (all nb 26)
- [ ] Alert if hit rate drops >50% week-over-week — that's a prompt-template regression

## Recap

1. **Agent prefill is quadratic in turns.** This is the dominant cost in agentic products and it is
   almost entirely avoidable.
2. **Prompt layout decides your cache hit rate.** One volatile token at the top costs you everything.
3. **Reusing KV beats recomputing it** at nearly every prefix size over PCIe — enable offload.
4. **Cascades have a break-even escalation rate.** Above it, you're paying twice for a worse answer.
5. **Semantic caching trades correctness for cost**, and the failure is silent. Prefix caching gives
   you most of the win with none of the risk, because it reuses *computation*, never *conclusions*.

### Further reading
- [LMCache](https://github.com/LMCache/LMCache) · [Mooncake](https://arxiv.org/abs/2407.00079) — KV as shared infrastructure
- [SGLang / RadixAttention](https://arxiv.org/abs/2312.07104) — prefix sharing as a first-class primitive
- [FrugalGPT](https://arxiv.org/abs/2305.05176) — the cascade idea, priced
- Prerequisites: nb [22](./vLLM_High_Throughput_Serving.ipynb) (prefix caching), [26](./Serving_Logs_Observability.ipynb) (hit-rate metrics), [29](./Distributed_MultiReplica_Serving.ipynb) (routing)